# Paper results: figures and tables

This notebook builds the paper's main-results figures and tables from the per-pixel
prediction CSVs in `results/`. Each section below produces one paper element, in the
order they appear in the paper, and writes it to `paper_latex/Images/` (figures) or
`paper_latex/Tables/` (tables).

| Section | Paper element | Output |
|---|---|---|
| A | Figure 2 — MAE vs. input months, plus inference time | `Images/performance_vs_months.pdf` |
| B | Table 1 — Per-tile MAE | `Tables/tile_performance.tex` |
| C | Figure 3 — Qualitative per-tile predictions | `Images/qualitative_*.pdf`, `Figures/qualitative_tiles.tex` |
| D | Table 2 — Pretraining × time/location ablation | `Tables/ablation.tex` |
| E | Table 3 — Presto regularization ablation | `Tables/presto_locdrop_ablation.tex` |
| F | Table 4 — Prithvi raw-input concatenation ablation | `Tables/prithvi_concat_ablation.tex` |
| G | Table 5 — Prithvi model size ablation | `Tables/prithvi_size_ablation.tex` |
| H | Table 6 — Ensemble comparison | `Tables/ensemble_comparison.tex` |
| I | Figure 7 — Prithvi crop size | `Images/crop_size_effect.pdf` |
| J | Figure 8 — MAE vs. number of missing timesteps | `Images/missing_timesteps.pdf` |

Figures 4–6 (ecoregion, land cover, spatial variation) are produced by `supportive-analysis/`.

**How MAE is computed everywhere below** (unless a section says otherwise):
1. For one seed, take the absolute error per pixel and phenophase, average it within each
   test tile-year (site × year), then average equally over the 48 test tile-years.
2. "Mean" is the average of the four phenophases (greenup, maturity, senescence, dormancy).
3. Report the mean over seeds 42, 123, and 456; "±" and error bars are the sample standard
   deviation (ddof=1) across those seeds.

Run from the repository root after evaluation (`eval_to_dataframe.py`) and ensembling
(`run_jobs/run_ensembles.sh`). The qualitative figure additionally needs the cache from
`misc_scripts/build_qualitative_cache.py`. Clear outputs before committing.


## Setup

Imports and shared constants.


In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import os
from lib.utils_plotting import results_file, sort_df

sns.set_theme()

# Date display names and column prefix mapping
DATE_NAMES = ["Greenup", "Maturity", "Senescence", "Dormancy"]
DATE_SHORT = ["G", "M", "S", "D"]  # column prefixes in CSVs

MONTH_SUBSETS = {
    4: [3, 6, 9, 12],
    8: [3, 4, 5, 6, 7, 8, 9, 10],
    12: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
}


### Load results

For each month setting, load every model group's per-pixel test predictions for the three
seeds (`all_results_per_seed[T][group]`, one DataFrame per seed). Seed-ensemble groups
store their blended predictions as `seed_all_test.csv` and are loaded too. The dense
prior-work predictions (T = 244) are the same for every T.

`all_mae_dfs[T]` holds per-tile-year MAE (averaged over seeds) for every group; it feeds
the prior-work reference line in Figure 2 and the crop-size figure.


In [ ]:
# Dense prior work (T=244): same test pixels for every month subset
PRIOR_WORK_KEY = "transformer_1d_priorwork_data_test"
prior_work_df = sort_df(pd.read_csv(os.path.join("results", f"{PRIOR_WORK_KEY}.csv")))

all_results = {}
all_mae_dfs = {}
all_results_per_seed = {}
CANONICAL_SEEDS = ["seed_42", "seed_123", "seed_456"]

for n_months, months in MONTH_SUBSETS.items():
    print(f"Loading {n_months}-month subset...", end=" ")
    res_test, res_per_seed_test = results_file("test", selected_months=months, seeds=CANONICAL_SEEDS)
    res_test[PRIOR_WORK_KEY] = prior_work_df
    all_results[n_months] = res_test
    all_results_per_seed[n_months] = res_per_seed_test

    # Per-tile-year MAE averaged across seeds (used for the prior-work line and crop-size plot)
    mae_dict = {"HLStile": [], "Date": [], "MAE": [], "Method": [], "SiteID": [], "years": []}
    ref_key = list(res_test.keys())[0]
    tile_info = res_test[ref_key][["HLStile", "SiteID", "years"]].drop_duplicates().values

    for key in res_test.keys():
        seed_dfs = res_per_seed_test.get(key, [res_test[key]])
        for tile, siteid, year in tile_info:
            seed_tile_maes = {d: [] for d in DATE_SHORT}
            for sdf in seed_dfs:
                df_tile = sdf[(sdf["HLStile"] == tile) & (sdf["SiteID"] == siteid) & (sdf["years"] == year)]
                if len(df_tile) == 0:
                    continue
                for d in DATE_SHORT:
                    seed_tile_maes[d].append(np.mean(np.abs(df_tile[f"{d}_pred_DOY"] - df_tile[f"{d}_truth_DOY"])))
            for d_short, d_name in zip(DATE_SHORT, DATE_NAMES):
                if seed_tile_maes[d_short]:
                    mae_dict["HLStile"].append(tile)
                    mae_dict["Date"].append(d_name)
                    mae_dict["MAE"].append(np.mean(seed_tile_maes[d_short]))
                    mae_dict["SiteID"].append(siteid)
                    mae_dict["years"].append(year)
                    mae_dict["Method"].append(key)

    all_mae_dfs[n_months] = pd.DataFrame(mae_dict)
    print("done.")

print("All subsets loaded.")


### Shared helpers

- `seed_stats` / `per_seed_stats` / `agg_seeds`: the MAE aggregation described at the top.
- `fmt_mean_std`, `mark`, `rank_marks`, `stat_cells`, `latex_table`, `write_table`: build the
  LaTeX tables (bold = best, underline = second best) and write them to `paper_latex/Tables/`.
- `render_latex_preview`: compiles a table snippet and shows it inline (needs LaTeX).


In [ ]:
# ---- Shared aggregation helpers: tile-year MAE -> per seed -> mean/std across seeds ----
def seed_stats(sdf):
    """Tile-year-balanced MAE per phenophase, plus their mean, for one seed's predictions."""
    err = pd.DataFrame({d_name: (sdf[f"{d}_pred_DOY"] - sdf[f"{d}_truth_DOY"]).abs()
                        for d_name, d in zip(DATE_NAMES, DATE_SHORT)})
    tile_means = err.groupby([sdf["HLStile"], sdf["SiteID"], sdf["years"]]).mean()
    stats = {d: float(tile_means[d].mean()) for d in DATE_NAMES}
    stats["Mean"] = float(np.mean([stats[d] for d in DATE_NAMES]))
    return stats


def per_seed_stats(per_seed_dict, keys):
    """One seed_stats dict per seed CSV, pooled over one or more result keys."""
    if isinstance(keys, str):
        keys = [keys]
    return [seed_stats(sdf) for key in keys for sdf in per_seed_dict.get(key, [])]


def agg_seeds(samples):
    """Mean over seeds for every column, plus the across-seed std (ddof=1) of Mean."""
    if not samples:
        return None
    agg = {c: float(np.mean([s[c] for s in samples])) for c in DATE_NAMES + ["Mean"]}
    means = [s["Mean"] for s in samples]
    agg["Mean_std"] = float(np.std(means, ddof=1)) if len(means) > 1 else 0.0
    return agg


def fmt_mean_std(m, sd):
    return f"{m:.1f}{{\\scriptsize\\,$\\pm${sd:.1f}}}"


def mark(txt, rank):
    """Wrap a formatted cell in bold (best) or underline (second best)."""
    return {"best": r"\textbf{" + txt + "}", "second": r"\underline{" + txt + "}"}.get(rank, txt)


def rank_marks(values, n=2):
    """Map the keys of the n smallest values to "best"/"second" (stable order, NaNs ignored)."""
    ranked = sorted(((k, v) for k, v in values.items() if not np.isnan(v)), key=lambda kv: kv[1])
    return {k: r for (k, _), r in zip(ranked, ["best", "second"][:n])}


def stat_cells(agg, ranks=None):
    """Formatted G, M, S, D, and Mean ± std cells for one table row."""
    if agg is None:
        return ["---"] * 5
    ranks = ranks or {}
    cells = [mark(f"{agg[d]:.1f}", ranks.get(d, "")) for d in DATE_NAMES]
    cells.append(mark(fmt_mean_std(agg["Mean"], agg["Mean_std"]), ranks.get("Mean", "")))
    return cells


def latex_table(label, caption, colspec, header, groups, footer=None):
    """Single-column booktabs table; row groups are separated by midrules."""
    L = [r"\begin{table}[!t]", r"\centering", r"\footnotesize", r"\setlength{\tabcolsep}{4pt}",
         r"\caption{" + caption + "}", r"\label{" + label + "}",
         r"\begin{tabular}{" + colspec + "}", r"\toprule", header + r" \\", r"\midrule"]
    for gi, rows in enumerate(groups):
        if gi > 0:
            L.append(r"\midrule")
        L += [" & ".join(row) + r" \\" for row in rows]
    if footer:
        L.append(r"\midrule")
        L += [" & ".join(row) + r" \\" for row in footer]
    L += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    return "\n".join(L)


def write_table(latex_str, filename):
    """Preview the table inline and write it to paper_latex/Tables/<filename>."""
    render_latex_preview(latex_str)
    with open(f"paper_latex/Tables/{filename}", "w") as f:
        f.write(latex_str)
    print(f"Saved: paper_latex/Tables/{filename}")


CHECK = {True: r"\checkmark", False: r"$\times$"}


In [ ]:
import subprocess, tempfile, os
from IPython.display import display, Image

def render_latex_preview(latex_str, dpi=200):
    """Compile a LaTeX snippet to PDF, convert to PNG, and display inline."""
    doc = r"""\documentclass[final,3p,times,twocolumn]{elsarticle}
\usepackage{amssymb}
\usepackage{amsmath}
\usepackage{booktabs}
\usepackage{multirow}
\makeatletter
\def\@author{}
\def\@title{}
\makeatother
\begin{document}
""" + latex_str + r"""
\end{document}"""
    
    with tempfile.TemporaryDirectory() as tmpdir:
        # Copy elsarticle.cls to tmpdir
        import shutil
        cls_src = os.path.join("paper_latex", "elsarticle.cls")
        if os.path.exists(cls_src):
            shutil.copy(cls_src, tmpdir)
        
        tex_path = os.path.join(tmpdir, "preview.tex")
        with open(tex_path, "w") as f:
            f.write(doc)
        cmd = f"module load texlive/2023 && pdflatex -interaction=nonstopmode -output-directory {tmpdir} {tex_path}"
        result = subprocess.run(cmd, shell=True, executable="/bin/bash", capture_output=True, text=True)
        pdf_path = os.path.join(tmpdir, "preview.pdf")
        if not os.path.exists(pdf_path):
            print("LaTeX compilation failed:")
            # Show just the error
            for line in result.stdout.split('\n'):
                if line.startswith('!') or 'Error' in line:
                    print(line)
            return
        png_path = os.path.join(tmpdir, "preview")
        subprocess.run(["pdftoppm", "-png", "-r", str(dpi), pdf_path, png_path], capture_output=True)
        png_file = png_path + "-1.png"
        if os.path.exists(png_file):
            display(Image(filename=png_file))
            # Also save PDF copy for inspection
            import shutil
            shutil.copy(pdf_path, "paper_latex/Images/_preview.pdf")
        else:
            print("PNG conversion failed")


## A. Figure 2 — MAE vs. number of input months (`fig:performance_vs_months`)

Grouped bars: MAE per phenophase and their mean for the Temporal Transformer, Presto,
Prithvi, and the all-model ensemble at T = 4, 8, 12 (aggregation as described at the top;
error bars = std across seeds). The dashed line is the dense prior work (T = 244), computed
the same way on the same test pixels. The efficiency panel plots mean per-tile inference
time from `results/benchmark_results.csv`; the T = 244 line is the Temporal Transformer
architecture run on 244 timesteps.


In [ ]:
methods_compare = {
    "Temporal Transformer": "transformer_1d_paper_nl3_1.0_test",
    "Presto": "presto_1.0_test",
    "Prithvi": "prithvi_final_100m_crop32_1.0_test",
    "Ensemble (all)": "ensemble_all_test",
}

dates = ["Greenup", "Maturity", "Senescence", "Dormancy"]
n_months_list = sorted(MONTH_SUBSETS.keys())

# Per-seed tile-year-balanced MAE; bars = mean across seeds, error bars = std (ddof=1)
rows = []
for n_months in n_months_list:
    per_seed = all_results_per_seed.get(n_months, {})
    for name, key in methods_compare.items():
        samples = per_seed_stats(per_seed, key)
        if not samples:
            continue
        for d in DATE_NAMES + ["Mean"]:
            vals = [s[d] for s in samples]
            rows.append({"Months": n_months, "Method": name, "Date": d,
                         "MAE": np.mean(vals), "Std": np.std(vals, ddof=1) if len(vals) > 1 else 0.0})

comp_df = pd.DataFrame(rows)

# Prior work MAE (fixed reference — same data for all subsets)
prior_mae = {}
mdf_ref = all_mae_dfs[n_months_list[-1]]
for d in dates:
    prior_mae[d] = mdf_ref[(mdf_ref["Method"] == PRIOR_WORK_KEY) & (mdf_ref["Date"] == d)]["MAE"].mean()
prior_mae["Mean"] = mdf_ref[mdf_ref["Method"] == PRIOR_WORK_KEY]["MAE"].mean()

# --- Load benchmark results ---
bench_df = pd.read_csv("results/benchmark_results.csv")
bench_mean = bench_df.groupby(["Model", "Timesteps"])["Time (s)"].mean().reset_index()

bench_to_label = {
    "1D Transformer": "Temporal Transformer",
    "Prithvi (100M, crop32)": "Prithvi",
    "Presto": "Presto",
    "Ensemble (all)": "Ensemble (all)",
}


In [ ]:
# --- Plot: 2x3 layout (G, M / S, D / Mean, Efficiency) ---
from matplotlib.gridspec import GridSpec

sns.set_theme(style="whitegrid")

COL_GAP = 0.01
fig = plt.figure(figsize=(19, 7))
gs = GridSpec(2, 4, figure=fig, width_ratios=[1, 1, COL_GAP, 1],
              wspace=0.18, hspace=0.5)

axes = np.empty((2, 3), dtype=object)
axes[0, 0] = fig.add_subplot(gs[0, 0])
axes[0, 1] = fig.add_subplot(gs[0, 1])
axes[1, 0] = fig.add_subplot(gs[1, 0])
axes[1, 1] = fig.add_subplot(gs[1, 1])
axes[0, 2] = fig.add_subplot(gs[0, 3])
# Efficiency panel: narrower than its cell, centered within it
eff_outer = gs[1, 3].get_position(fig)
eff_shrink = 0.35
eff_w = eff_outer.width * (1 - eff_shrink)
eff_x = eff_outer.x0 + (eff_outer.width - eff_w) / 2
axes[1, 2] = fig.add_axes([eff_x, eff_outer.y0, eff_w, eff_outer.height])

panel_positions = [(0,0), (0,1), (1,0), (1,1), (0,2)]
colors = sns.color_palette("Set2", len(methods_compare))
method_names = list(methods_compare.keys())
n_methods = len(method_names)
group_spacing = 1.05
x = np.arange(len(n_months_list)) * group_spacing
bar_width_frac = 0.97
bar_width = bar_width_frac / n_methods

for idx, d in enumerate(dates + ["Mean"]):
    ax = axes[panel_positions[idx]]
    # Panel y-range floats with data
    panel_vals, panel_stds = [], []
    for name in method_names:
        sub = comp_df[(comp_df["Method"] == name) & (comp_df["Date"] == d)]
        for m in n_months_list:
            panel_vals.append(sub[sub["Months"] == m]["MAE"].values[0])
            panel_stds.append(sub[sub["Months"] == m]["Std"].values[0])
    y_lo = min(panel_vals) - 0.8
    y_hi = max(v + s for v, s in zip(panel_vals, panel_stds)) + 0.6

    for i, name in enumerate(method_names):
        sub = comp_df[(comp_df["Method"] == name) & (comp_df["Date"] == d)]
        vals = [sub[sub["Months"] == m]["MAE"].values[0] for m in n_months_list]
        stds = [sub[sub["Months"] == m]["Std"].values[0] for m in n_months_list]
        is_ensemble = (name == "Ensemble (all)")
        edge_width = 1.4 if is_ensemble else 0.7
        bars = ax.bar(x + i * bar_width - (n_methods - 1) * bar_width / 2, vals,
                      bar_width, label=name if idx == 0 else None, color=colors[i],
                      yerr=stds, capsize=2.5, error_kw={"linewidth": 0.9},
                      edgecolor="black", linewidth=edge_width, zorder=3)
    ax.axhline(y=prior_mae[d], color="#444444", linestyle=(0, (3, 2)), linewidth=1.8,
               label="Temporal Transformer (T = 244)" if idx == 0 else None, zorder=2)

    ax.set_title(d, fontsize=14, fontweight="bold" if d == "Mean" else "normal")
    ax.set_xlabel("# Input Months")
    ax.set_xticks(x)
    ax.set_xticklabels(n_months_list)
    half_span = bar_width * n_methods / 2
    ax.set_xlim(x[0] - half_span - 0.02, x[-1] + half_span + 0.02)
    ax.set_ylim(y_lo, y_hi)
    if idx == 0:
        ax.set_ylabel("MAE (days)")
    # Fine-grained y grid: major (labelled) every 0.5, minor every 0.25
    ax.yaxis.set_major_locator(plt.MultipleLocator(0.5))
    ax.yaxis.set_minor_locator(plt.MultipleLocator(0.25))
    ax.grid(axis="y", which="major", linestyle="-", linewidth=0.8, alpha=0.5, color="#888888")
    ax.grid(axis="y", which="minor", linestyle=":", linewidth=0.5, alpha=0.4, color="#aaaaaa")
    ax.tick_params(axis="y", which="major", labelsize=9)
    ax.set_axisbelow(True)

# Efficiency panel (bottom-right, narrower)
ax = axes[1, 2]
bench_colors = {
    "Temporal Transformer": colors[0],
    "Presto":               colors[1],
    "Prithvi":              colors[2],
    "Ensemble (all)":       colors[3],
}
bench_markers = {"Temporal Transformer": "s", "Prithvi": "o", "Presto": "D", "Ensemble (all)": "^"}

for model_name in bench_mean["Model"].unique():
    label = bench_to_label.get(model_name, model_name)
    if label not in bench_colors:
        continue
    sub = bench_mean[bench_mean["Model"] == model_name].sort_values("Timesteps")
    sub = sub[sub["Timesteps"].isin([4, 8, 12])]
    if len(sub) == 0:
        continue
    ax.plot(sub["Timesteps"], sub["Time (s)"], marker=bench_markers[label],
            color=bench_colors[label], linewidth=2, markersize=7, label=label)

prior_bench = bench_mean[(bench_mean["Model"] == "1D Transformer") & (bench_mean["Timesteps"] == 244)]
if len(prior_bench) > 0:
    prior_time_val = prior_bench["Time (s)"].values[0]
    ax.axhline(y=prior_time_val, color="#444444", linestyle=(0, (3, 2)), linewidth=1.8, zorder=2)
    ax.text(ax.get_xlim()[0] + 0.5, prior_time_val + 0.08, "T = 244",
            color="#444444", fontsize=9, fontweight="bold", va="bottom")

ax.set_title("Efficiency", fontsize=14, fontweight="bold")
ax.set_xlabel("# Input Timesteps")
ax.set_xticks(n_months_list)
ax.set_ylabel("Inference Time / Tile (s)")
ax.yaxis.set_major_locator(plt.MultipleLocator(0.5))
ax.yaxis.set_minor_locator(plt.MultipleLocator(0.1))
ax.grid(axis="y", which="major", linestyle="-", linewidth=0.8, alpha=0.5, color="#888888")
ax.grid(axis="y", which="minor", linestyle=":", linewidth=0.5, alpha=0.4, color="#aaaaaa")
ax.set_axisbelow(True)

handles, labels = axes[0, 0].get_legend_handles_labels()
h6, l6 = axes[1, 2].get_legend_handles_labels()
seen = set(labels)
for h, l in zip(h6, l6):
    if l not in seen:
        handles.append(h)
        labels.append(l)
        seen.add(l)

fig.legend(handles, labels, loc="lower center", ncol=len(handles),
           bbox_to_anchor=(0.5, -0.08), fontsize=14)
plt.show()


In [ ]:
# Save figure
fig.savefig("paper_latex/Images/performance_vs_months.pdf", bbox_inches="tight", dpi=300)
print("Saved: paper_latex/Images/performance_vs_months.pdf")


## B. Table 1 — Per-tile MAE (`tab:tile_performance`)

Six illustrative test sites. Unlike the rest of the notebook, MAE here is pooled over all
sampled pixels of a site across both years (2019 and 2020), per seed, then averaged over
seeds; the Mean column also shows the std across seeds. The top row of each sub-table is
the dense prior work (T = 244), computed the same way and excluded from the bold/underline
ranking, which is applied to the Mean column only.


In [ ]:
heatmap_methods = {"Trans":"transformer_1d_paper_nl3_1.0_test","Presto":"presto_1.0_test",
                   "Prithvi":"prithvi_final_100m_crop32_1.0_test",
                   "Ens.":"ensemble_all_test"}
model_names = list(heatmap_methods.keys())
month_subsets_sorted = sorted(MONTH_SUBSETS.keys())
month_names = {4:"4", 8:"8", 12:"12"}

selected_tiles = [
    # Illustrative tiles across the 4/8/12-month subsets. MD-3 and WA-2 were
    # selected jointly as the unused pair closest to the test-set-average profile
    # of baseline difficulty and Presto, Prithvi, and ensemble improvements.
    ("CO-3","T13TDE"),("AZ-6","T12SWA"),("KS-1","T15SUD"),
    ("NV-2","T11SQD"),("MD-3","T18SUJ"),("WA-2","T10TER"),
]


def _per_seed_tile_stats(seed_dfs, siteid, hls_tile):
    """For a given (SiteID, HLStile), compute per-seed (G,M,S,D,Mean) MAE."""
    out = []
    for sdf in seed_dfs:
        df_t = sdf[(sdf["SiteID"] == siteid) & (sdf["HLStile"] == hls_tile)]
        if len(df_t) == 0:
            continue
        sm = {}
        for d_name, d_short in zip(DATE_NAMES, DATE_SHORT):
            sm[d_name] = float(np.mean(np.abs(
                df_t[f"{d_short}_pred_DOY"] - df_t[f"{d_short}_truth_DOY"])))
        sm["Mean"] = float(np.mean([sm[d] for d in DATE_NAMES]))
        out.append(sm)
    return out


# Pre-compute per-(siteid, tile, T, model) aggregate (mean for each col, std for Mean)
# tile_data[(siteid, tile)][(model, nm)] = {date: mean, "Mean": mean, "Mean_std": std}
tile_data = {}
for nm in month_subsets_sorted:
    per_seed = all_results_per_seed.get(nm, {})
    for ml, mk in heatmap_methods.items():
        seed_dfs = per_seed.get(mk, [])
        if not seed_dfs:
            continue
        for siteid, hls_tile in selected_tiles:
            samples = _per_seed_tile_stats(seed_dfs, siteid, hls_tile)
            if not samples:
                continue
            agg = {d: float(np.mean([s[d] for s in samples])) for d in DATE_NAMES}
            means = np.array([s["Mean"] for s in samples])
            agg["Mean"] = float(means.mean())
            agg["Mean_std"] = float(means.std(ddof=1)) if len(means) > 1 else 0.0
            k = (siteid, hls_tile)
            if k not in tile_data: tile_data[k] = {}
            tile_data[k][(ml, nm)] = agg

# Prior work: same across month subsets, no std (single reference run)
# Pool both years per tile, exactly like the per-seed model rows above.
prior_tile_data = {}
for siteid, hls_tile in selected_tiles:
    df_t = prior_work_df[(prior_work_df["SiteID"] == siteid) & (prior_work_df["HLStile"] == hls_tile)]
    if len(df_t) == 0:
        continue
    prior_tile_data[(siteid, hls_tile)] = {
        d_name: float(np.mean(np.abs(df_t[f"{d_short}_pred_DOY"] - df_t[f"{d_short}_truth_DOY"])))
        for d_name, d_short in zip(DATE_NAMES, DATE_SHORT)
    }


In [ ]:
tiles_per_row = 3
L = []
L.append(r"\begin{table*}[!t]")
L.append(r"\centering")
L.append(r"\caption{Per-tile MAE (days). The top row in each sub-table is the prior-work temporal transformer (T{=}244) and is excluded from best/second-best ranking. The Mean column reports mean $\pm$ std across seeds. \textbf{Bold} = best, \underline{underline} = second best in the Mean column among the fine-tuned models.}")
L.append(r"\label{tab:tile_performance}")
L.append(r"\setlength{\tabcolsep}{3pt}")
L.append(r"\footnotesize")

for tile_row_start in range(0, len(selected_tiles), tiles_per_row):
    tile_row = selected_tiles[tile_row_start:tile_row_start+tiles_per_row]
    if tile_row_start > 0:
        L.append(r"")
        L.append(r"\vspace{8pt}")
        L.append(r"")
    for ti, (siteid, tile) in enumerate(tile_row):
        if ti > 0:
            L.append(r"\hfill")
        L.append(r"\begin{minipage}{0.31\textwidth}")
        L.append(r"\centering")
        L.append(r"{\bfseries " + siteid + "}")
        L.append(r"\vspace{1pt}")
        L.append("")
        L.append(r"\begin{tabular}{llrrrrr}")
        L.append(r"\toprule")
        L.append(r"$T$ & Model & G & M & S & D & Mean \\")
        L.append(r"\midrule")

        # Prior-work row (no std, no ranking)
        prior_vals = prior_tile_data.get((siteid, tile), {d: float("nan") for d in DATE_NAMES})
        prior_list = [prior_vals[d] for d in DATE_NAMES]
        prior_valid = [v for v in prior_list if not np.isnan(v)]
        prior_mean = np.mean(prior_valid) if prior_valid else float("nan")
        prior_cells = ["244", "Trans"]
        for v in prior_list + [prior_mean]:
            prior_cells.append("---" if np.isnan(v) else f"{v:.1f}")
        L.append(" & ".join(prior_cells) + r" \\")
        L.append(r"\specialrule{1.2pt}{0pt}{0pt}")

        for ni, nm in enumerate(month_subsets_sorted):
            # Build row dicts for ranking on Mean
            mean_ci = 4
            model_vals = {}
            for model in model_names:
                cell_agg = tile_data.get((siteid, tile), {}).get((model, nm))
                if cell_agg is None:
                    model_vals[model] = ([float("nan")] * 4 + [float("nan")], 0.0)
                else:
                    vals = [cell_agg[d] for d in DATE_NAMES] + [cell_agg["Mean"]]
                    model_vals[model] = (vals, cell_agg["Mean_std"])

            # Best/second only for Mean column (ci=4) among finetuned models
            fmt = {}
            mean_pairs = [(m, model_vals[m][0][mean_ci]) for m in model_names
                          if not np.isnan(model_vals[m][0][mean_ci])]
            mean_pairs.sort(key=lambda x: x[1])
            if len(mean_pairs) >= 1: fmt[(mean_pairs[0][0], mean_ci)] = "best"
            if len(mean_pairs) >= 2: fmt[(mean_pairs[1][0], mean_ci)] = "second"

            for mi, model in enumerate(model_names):
                vals, mean_std = model_vals[model]
                cells = [month_names[nm] if mi == 0 else "", model]
                for ci in range(5):
                    v = vals[ci]
                    if np.isnan(v):
                        cells.append("---")
                        continue
                    if ci == mean_ci:
                        txt = fmt_mean_std(v, mean_std)
                    else:
                        txt = f"{v:.1f}"
                    rank = fmt.get((model, ci), "")
                    if rank == "best":
                        cells.append(r"\textbf{" + txt + "}")
                    elif rank == "second":
                        cells.append(r"\underline{" + txt + "}")
                    else:
                        cells.append(txt)
                L.append(" & ".join(cells) + r" \\")

            if ni < len(month_subsets_sorted) - 1:
                L.append(r"\midrule")

        L.append(r"\bottomrule")
        L.append(r"\end{tabular}")
        L.append(r"\end{minipage}")

L.append(r"\end{table*}")

latex_str = "\n".join(L)
write_table(latex_str, "tile_performance.tex")


## C. Figure 3 — Qualitative per-tile predictions (`fig:qualitative_tiles`)

Maps for the four tiles in `evaluation.qualitative_tiles` (`config.json`), 4-month setting,
seed 42. Rows: HP-LSP reference, Temporal Transformer, Prithvi, and the all-model ensemble
(seed-42 ensemble weights); columns: HLS RGB plus the four phenophases. Each phenophase
column's color range is the 5th–95th percentile of the valid reference values, shared by
all rows. Predictions are read from the cache written by
`misc_scripts/build_qualitative_cache.py` (stored as DOY / 547).


In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.gridspec import GridSpec
import path_config
path_config._PATHS_CACHE = None

qual_months = [3, 6, 9, 12]
qual_tiles = path_config.get_qual_tile_ids()
cache_dir = os.path.join(path_config.get_qual_cache_dir(),
                         f"m{'-'.join(str(m) for m in qual_months)}")

PHASE_NAMES = ["Greenup", "Maturity", "Senescence", "Dormancy"]
DOY_SCALE = 547
PCT_LO, PCT_HI = 5, 95
HEADER_FS = 22

TRANS_KEY    = "transformer_1d_paper_nl3_1.0"
PRITHVI_KEY  = "prithvi_final_100m_crop32_1.0"
ENSEMBLE_KEY = "__ensemble__"

missing = [f"{sid}_{htile}" for sid, htile in qual_tiles
           if not os.path.exists(os.path.join(cache_dir, f"{sid}_{htile}.npz"))]
if missing:
    raise FileNotFoundError(
        f"Missing cache files in {cache_dir}: {missing}\n"
        f"Run `python misc_scripts/build_qualitative_cache.py` on a GPU node."
    )

cache = {}
for sid, htile in qual_tiles:
    c = np.load(os.path.join(cache_dir, f"{sid}_{htile}.npz"), allow_pickle=False)
    names = [str(n) for n in c["model_names"]]
    preds = {name: c["predictions"][i] for i, name in enumerate(names)}
    cache[(sid, htile)] = {
        "gt": c["ground_truth"],
        "hls_rgb": c["hls_rgb"],
        "preds": preds,
    }


In [ ]:
# Match the reference paper's rainbow scale while giving midrange greens
# more visual width than standard jet. Scalar limits remain unchanged.
_cmap_positions = np.linspace(0.0, 1.0, 256)
_jet_positions = np.interp(
    _cmap_positions,
    [0.00, 0.22, 0.34, 0.66, 0.78, 1.00],
    [0.00, 0.30, 0.42, 0.58, 0.70, 1.00],
)
cmap_pred = LinearSegmentedColormap.from_list(
    "stretched_jet", plt.cm.jet(_jet_positions), N=256
)
bad_color = "0.85"


def _apply_clip_ticks(cbar, vmin, vmax, labelsize=14):
    cbar.ax.xaxis.set_tick_params(labelsize=labelsize)
    mid = (vmin + vmax) / 2
    cbar.set_ticks([vmin, mid, vmax])
    cbar.set_ticklabels([f"$\\leq${vmin:.0f}", f"{mid:.0f}", f"$\\geq${vmax:.0f}"])


pdf_paths = []
for sid, htile in qual_tiles:
    entry = cache[(sid, htile)]
    gt = entry["gt"]
    invalid = gt[0] == -1
    rgb = entry["hls_rgb"].copy()
    rgb[invalid] = 0.85

    for k in (TRANS_KEY, PRITHVI_KEY, ENSEMBLE_KEY):
        if k not in entry["preds"]:
            raise KeyError(f"Cache for {sid}/{htile} missing predictions for '{k}'. "
                           f"Rebuild with --models including transformer + prithvi + ensemble.")
    pred_T = entry["preds"][TRANS_KEY]
    pred_P = entry["preds"][PRITHVI_KEY]
    pred_E = entry["preds"][ENSEMBLE_KEY]

    n_rows, n_cols = 4, 1 + len(PHASE_NAMES)
    fig = plt.figure(figsize=(n_cols * 3.2, n_rows * 3.2 + 1.0))
    gs = GridSpec(n_rows + 1, n_cols, figure=fig,
                  height_ratios=[1] * n_rows + [0.08],
                  hspace=0.03, wspace=0.05)

    # Per-column ranges from valid HP-LSP values; apply them to every row.
    col_pred_ranges = [None]
    for pi in range(len(PHASE_NAMES)):
        masked = np.ma.masked_where(invalid, gt[pi] * DOY_SCALE)
        all_vals = masked.compressed()
        if all_vals.size == 0:
            vmin, vmax = 0.0, 1.0
        else:
            vmin = float(np.percentile(all_vals, PCT_LO))
            vmax = float(np.percentile(all_vals, PCT_HI))
            if vmin == vmax:
                vmax = vmin + 1.0
        col_pred_ranges.append((vmin, vmax))

    # Row order: GT (top), Trans, Prithvi, Ensemble
    row_specs = [
        ("HP-LSP",      gt),
        ("Temp Trans.", pred_T),
        ("Prithvi",     pred_P),
        ("Ensemble",    pred_E),
    ]

    last_pred_im = [None] * n_cols
    for ri, (lbl, src) in enumerate(row_specs):
        # Col 0: HLS RGB (repeated)
        ax = fig.add_subplot(gs[ri, 0])
        ax.imshow(rgb)
        ax.set_xticks([]); ax.set_yticks([])
        if ri == 0:
            ax.set_title(f"{sid} / {htile}", fontsize=HEADER_FS, fontweight="bold")
        ax.set_ylabel(lbl, fontsize=HEADER_FS, fontweight="bold")

        for pi, pname in enumerate(PHASE_NAMES):
            col = 1 + pi
            ax = fig.add_subplot(gs[ri, col])
            ax.set_xticks([]); ax.set_yticks([])
            data = src[pi] * DOY_SCALE
            vmin, vmax = col_pred_ranges[col]
            cm = cmap_pred.copy(); cm.set_bad(bad_color)
            im = ax.imshow(np.clip(np.ma.masked_where(invalid, data), vmin, vmax),
                           vmin=vmin, vmax=vmax, cmap=cm)
            last_pred_im[col] = im
            if ri == 0:
                ax.set_title(pname, fontsize=HEADER_FS, fontweight="bold")

    # One colorbar per phenophase column at the bottom
    for col in range(1, n_cols):
        outer = gs[n_rows, col].get_position(fig)
        cbar_w = outer.width * 0.78
        cbar_x = outer.x0 + (outer.width - cbar_w) / 2
        cax = fig.add_axes([cbar_x, outer.y0, cbar_w, outer.height])
        cbar = fig.colorbar(last_pred_im[col], cax=cax,
                            orientation="horizontal", extend="both")
        vmin, vmax = col_pred_ranges[col]
        _apply_clip_ticks(cbar, vmin, vmax, labelsize=14)
        if col == 1:
            cbar.set_label("DOY", fontsize=15)
    ax = fig.add_subplot(gs[n_rows, 0]); ax.axis("off")

    out_path = f"paper_latex/Images/qualitative_{sid}_{htile}.pdf"
    fig.savefig(out_path, bbox_inches="tight", dpi=200)
    print(f"Saved: {out_path}")
    pdf_paths.append((sid, htile, out_path))
    plt.show()


In [ ]:
# ---- Emit a 2x2 LaTeX figure stacking the four per-tile PDFs ----
assert len(pdf_paths) == 4, f"Expected 4 tiles for 2x2 layout, got {len(pdf_paths)}"

def _rel(p):
    return p.replace("paper_latex/", "")

rows = []
rows.append(r"\begin{figure*}[!t]")
rows.append(r"\centering")
for i in range(0, 4, 2):
    rows.append(r"\begin{minipage}{0.49\textwidth}\centering")
    rows.append(r"  \includegraphics[width=\linewidth]{" + _rel(pdf_paths[i][2]) + "}")
    rows.append(r"\end{minipage}\hfill")
    rows.append(r"\begin{minipage}{0.49\textwidth}\centering")
    rows.append(r"  \includegraphics[width=\linewidth]{" + _rel(pdf_paths[i+1][2]) + "}")
    rows.append(r"\end{minipage}")
    if i == 0:
        rows.append(r"\\[6pt]")
rows.append(r"\caption{Qualitative per-tile predictions for four representative test tiles "
            r"from the 4-month subset, selected to illustrate differences between the Temporal "
            r"Transformer and Prithvi across phenophases and spatial patterns. Within each "
            r"sub-panel, rows are HP-LSP (Reference tiles), Temp Trans.\ (Temporal Transformer), Prithvi, and Ensemble; "
            r"columns are the HLS RGB composite (repeated) plus the four phenophases "
            r"(Greenup, Maturity, Senescence, Dormancy). Phenophase colormaps use the 5th/95th "
            r"percentiles of valid HP-LSP values per column, with the resulting scale shared "
            r"across all four rows. A stretched rainbow colormap expands the midrange green "
            r"interval while preserving the displayed DOY limits.}")
rows.append(r"\label{fig:qualitative_tiles}")
rows.append(r"\end{figure*}")

tex_path = "paper_latex/Figures/qualitative_tiles.tex"
os.makedirs(os.path.dirname(tex_path), exist_ok=True)
with open(tex_path, "w") as f:
    f.write("\n".join(rows) + "\n")
print(f"Saved: {tex_path}")


## D. Table 2 — Pretraining × time/location ablation, 4 months (`tab:ablation`)

Prithvi and Presto with and without pretrained weights and time/location inputs, plus the
Temporal Transformer as a reference row. Bold/underline mark the best/second best per
column within each model.


In [ ]:
ablation_variants = [
    # (model, result key, pretrained, time/location)
    ("Trans.",  "transformer_1d_paper_nl3_1.0_test",                  False, False),
    ("Prithvi", "prithvi_final_100m_crop32_1.0_test",                 True,  True),
    ("Prithvi", "prithvi_final_100m_crop32_notimeloc_1.0_test",       True,  False),
    ("Prithvi", "prithvi_nopretrain_100m_crop32_1.0_test",            False, True),
    ("Prithvi", "prithvi_nopretrain_100m_crop32_notimeloc_1.0_test",  False, False),
    ("Presto",  "presto_1.0_test",                                    True,  True),
    ("Presto",  "presto_notimeloc_1.0_test",                          True,  False),
    ("Presto",  "presto_nopretrain_1.0_test",                         False, True),
    ("Presto",  "presto_notimeloc_nopretrain_1.0_test",               False, False),
]
per_seed4 = all_results_per_seed[4]
ablation = [(model, pre, tl, agg_seeds(per_seed_stats(per_seed4, key)))
            for model, key, pre, tl in ablation_variants]


In [ ]:
groups = []
for model in dict.fromkeys(row[0] for row in ablation):           # models in table order
    members = [row for row in ablation if row[0] == model]
    ranks = {}
    if len(members) > 1:                                            # no ranking for a single row
        for col in DATE_NAMES + ["Mean"]:
            for i, r in rank_marks({i: m[3][col] for i, m in enumerate(members)}).items():
                ranks.setdefault(i, {})[col] = r
    rows = []
    for i, (_, pre, tl, agg) in enumerate(members):
        model_cell = r"\multirow{" + str(len(members)) + r"}{*}{" + model + "}" if i == 0 else ""
        rows.append([model_cell, CHECK[pre], CHECK[tl]] + stat_cells(agg, ranks.get(i)))
    groups.append(rows)

latex_str = latex_table(
    "tab:ablation",
    r"Joint pretraining $\times$ time/location ablation (4 months). \checkmark\ = component enabled; $\times$ = disabled. The Mean column reports mean $\pm$ std across seeds. \textbf{Bold} = best, \underline{underline} = second best per column (within model).",
    "lcc|rrrrr", "Model & Pretrain & Time/Loc & G & M & S & D & Mean", groups)
write_table(latex_str, "ablation.tex")


## E. Table 3 — Presto regularization ablation (`tab:presto_locdrop_ablation`)

Presto with (✓) and without (×) its regularization (dropout in the transformer blocks plus
location dropout) at each T. Bold marks the better value per column within each T.


In [ ]:
locdrop_variants = [(True, "presto_1.0_test"), (False, "presto_nolocdrop_nodropout_1.0_test")]
locdrop = {n_months: [(has_reg, agg_seeds(per_seed_stats(all_results_per_seed[n_months], key)))
                      for has_reg, key in locdrop_variants]
           for n_months in sorted(MONTH_SUBSETS)}


In [ ]:
groups = []
for n_months, members in locdrop.items():
    ranks = {}
    for col in DATE_NAMES + ["Mean"]:
        for i, r in rank_marks({i: agg[col] for i, (_, agg) in enumerate(members)}, n=1).items():
            ranks.setdefault(i, {})[col] = r
    rows = []
    for i, (has_reg, agg) in enumerate(members):
        t_cell = r"\multirow{" + str(len(members)) + r"}{*}{" + str(n_months) + "}" if i == 0 else ""
        rows.append([t_cell, CHECK[has_reg]] + stat_cells(agg, ranks.get(i)))
    groups.append(rows)

latex_str = latex_table(
    "tab:presto_locdrop_ablation",
    r"Presto regularization ablation: location dropout ($p_{\text{loc}}>0$) + weight dropout. \checkmark\ = both enabled; $\times$ = both disabled. Missing runs shown as ---. The Mean column reports mean $\pm$ std across seeds. \textbf{Bold} = best per column within each month subset.",
    "cc|rrrrr", "$T$ & Reg. & G & M & S & D & Mean", groups)
write_table(latex_str, "presto_locdrop_ablation.tex")


## F. Table 4 — Prithvi raw-input concatenation, 4 months (`tab:prithvi_concat_ablation`)

Prithvi (100M, 32×32 crops) with and without the raw 6-band input concatenated to the
upscaled ViT features. Bold/underline mark best/second per column.


In [ ]:
concat_variants = [(True, "prithvi_final_100m_crop32_1.0_test"),
                   (False, "prithvi_final_100m_crop32_noconcat_1.0_test")]
concat = [(has_concat, agg_seeds(per_seed_stats(per_seed4, key))) for has_concat, key in concat_variants]

ranks = {}
for col in DATE_NAMES + ["Mean"]:
    for i, r in rank_marks({i: agg[col] for i, (_, agg) in enumerate(concat)}).items():
        ranks.setdefault(i, {})[col] = r
rows = [[CHECK[has_concat]] + stat_cells(agg, ranks.get(i)) for i, (has_concat, agg) in enumerate(concat)]

latex_str = latex_table(
    "tab:prithvi_concat_ablation",
    r"Effect of raw-input concatenation on Prithvi (4 months, 100M, crop32). \checkmark\ = raw 6-band input concatenated with upscaled ViT features before the fusion head. The Mean column reports mean $\pm$ std across seeds. \textbf{Bold} = best, \underline{underline} = second best per column.",
    "c|rrrrr", "Concat Input & G & M & S & D & Mean", [rows])
write_table(latex_str, "prithvi_concat_ablation.tex")


## G. Table 5 — Prithvi model size, 4 months (`tab:prithvi_size_ablation`)

Prithvi tiny (5M), 100M, and 300M backbones with the same adaptation (32×32 crops).
Bold/underline mark best/second per column.


In [ ]:
size_variants = [("Prithvi (Tiny)", "prithvi_final_tiny_crop32_1.0_test"),
                 ("Prithvi (100M)", "prithvi_final_100m_crop32_1.0_test"),
                 ("Prithvi (300M)", "prithvi_final_300m_crop32_1.0_test")]
sizes = [(name, agg_seeds(per_seed_stats(per_seed4, key))) for name, key in size_variants]

ranks = {}
for col in DATE_NAMES + ["Mean"]:
    for i, r in rank_marks({i: agg[col] for i, (_, agg) in enumerate(sizes)}).items():
        ranks.setdefault(i, {})[col] = r
rows = [[name] + stat_cells(agg, ranks.get(i)) for i, (name, agg) in enumerate(sizes)]

latex_str = latex_table(
    "tab:prithvi_size_ablation",
    r"Prithvi model size ablation (4 months, crop32). The Mean column reports mean $\pm$ std across seeds. \textbf{Bold} = best, \underline{underline} = second best per column.",
    "lrrrrr", "Variant & G & M & S & D & Mean", [rows])
write_table(latex_str, "prithvi_size_ablation.tex")


## H. Table 6 — Ensemble comparison (`tab:ensemble_comparison`)

Mean MAE ± std at each T for: individual models; intra-model "×3" ensembles (three seeds of
one model blended; each row pools three disjoint blends gA/gB/gC, so the std is across
blends); and cross-model ensembles (one ensemble per seed). The Δ row is the all-model
ensemble minus the best individual model. Bold/underline mark best/second per column.


In [ ]:
ensemble_rows = {
    # row label -> result keys pooled into the per-seed (or per-blend) samples
    "Transformer":               ["transformer_1d_paper_nl3_1.0_test"],
    "Presto":                    ["presto_1.0_test"],
    "Prithvi":                   ["prithvi_final_100m_crop32_1.0_test"],
    "Transformer \\texttimes 3": [f"transformer_1d_paper_nl3_1.0_seed_ensemble_g{g}_test" for g in "ABC"],
    "Presto \\texttimes 3":      [f"presto_1.0_seed_ensemble_g{g}_test" for g in "ABC"],
    "Prithvi \\texttimes 3":     [f"prithvi_final_100m_crop32_1.0_seed_ensemble_g{g}_test" for g in "ABC"],
    "Transformer + Presto":      ["ensemble_transformer_presto_test"],
    "Transformer + Prithvi":     ["ensemble_transformer_prithvi_test"],
    "All Models":                ["ensemble_all_test"],
}
row_groups = [["Transformer", "Presto", "Prithvi"],
              ["Transformer \\texttimes 3", "Presto \\texttimes 3", "Prithvi \\texttimes 3"],
              ["Transformer + Presto", "Transformer + Prithvi", "All Models"]]
month_keys = sorted(MONTH_SUBSETS)

# ens[label][T] = aggregated stats (Mean and Mean_std are used)
ens = {label: {n_months: agg_seeds(per_seed_stats(all_results_per_seed[n_months], keys))
               for n_months in month_keys}
       for label, keys in ensemble_rows.items()}


In [ ]:
ranks = {n_months: rank_marks({label: ens[label][n_months]["Mean"] for label in ensemble_rows})
         for n_months in month_keys}
groups = [[[label] + [mark(fmt_mean_std(ens[label][n]["Mean"], ens[label][n]["Mean_std"]),
                           ranks[n].get(label, "")) for n in month_keys]
           for label in group]
          for group in row_groups]
best_individual = {n: min(ens[label][n]["Mean"] for label in row_groups[0]) for n in month_keys}
footer = [[r"$\Delta$ (vs.\ best indiv.)"]
          + [f"{ens['All Models'][n]['Mean'] - best_individual[n]:+.1f}" for n in month_keys]]

latex_str = latex_table(
    "tab:ensemble_comparison",
    r"Ensemble comparison (mean MAE $\pm$ std, days). Individual and cross-model rows aggregate over the same three seeds. Intra-model `$\times 3$' rows blend three seeds; variation is across three disjoint blends. \textbf{Bold} = best, \underline{underline} = second best per column.",
    "l|rrr", "Method & 4 mo & 8 mo & 12 mo", groups, footer=footer)
write_table(latex_str, "ensemble_comparison.tex")


## I. Figure 7 — Effect of Prithvi crop size, 4 months (`fig:crop_size`)

Prithvi trained on s × s crops, s ∈ {16, 32, 48, 96, 144, 224}. Bars: MAE per phenophase;
line and labels: the mean over phenophases. Values come from `all_mae_dfs[4]` (per-tile-year
MAE averaged over seeds), so they equal the seed-averaged MAE described at the top.


In [ ]:
mae_df = all_mae_dfs[4]

crop_methods = {
    16:  "prithvi_final_100m_crop16_1.0_test",
    32:  "prithvi_final_100m_crop32_1.0_test",
    48:  "prithvi_final_100m_crop48_1.0_test",
    96:  "prithvi_final_100m_crop96_1.0_test",
    144: "prithvi_final_100m_crop144_1.0_test",
    224: "prithvi_final_100m_crop224_1.0_test",
}

crop_dates = ["Greenup", "Maturity", "Senescence", "Dormancy"]
crop_sizes = sorted(crop_methods.keys())

mae_per_date = {d: [] for d in crop_dates}
mae_mean = []

for cs in crop_sizes:
    df_crop = mae_df[mae_df["Method"] == crop_methods[cs]]
    for d in crop_dates:
        mae_per_date[d].append(df_crop[df_crop["Date"] == d]["MAE"].mean())
    mae_mean.append(df_crop["MAE"].mean())


In [ ]:
# --- Match main-figure style ---
sns.set_theme(style="whitegrid")

n_methods = len(crop_dates)
group_spacing = 1.05
bar_width_frac = 0.78
bar_width = bar_width_frac / n_methods
x = np.arange(len(crop_sizes)) * group_spacing

colors_bar = sns.color_palette("Set2", n_methods)

# Column-friendly sizing: width matches legend below the plot
fig, ax = plt.subplots(figsize=(4.2, 2.6))

# Floating y-limits based on data
all_vals = [v for d in crop_dates for v in mae_per_date[d]] + list(mae_mean)
y_lo = min(all_vals) - 0.6
y_hi = max(all_vals) + 0.7

for i, d in enumerate(crop_dates):
    positions = x + i * bar_width - (n_methods - 1) * bar_width / 2
    ax.bar(positions, mae_per_date[d], bar_width,
           label=d, color=colors_bar[i],
           edgecolor="black", linewidth=0.5, zorder=3)

# Mean line across group centers
ax.plot(x, mae_mean, marker="o", color="black",
        linewidth=1.4, markersize=4, label="Mean", zorder=5)
for xv, mv in zip(x, mae_mean):
    ax.text(xv, mv + 0.18, f"{mv:.1f}", ha="center", va="bottom",
            fontsize=8, fontweight="bold", color="black", zorder=6,
            bbox=dict(boxstyle="round,pad=0.15", facecolor="white",
                      edgecolor="black", linewidth=0.5))

ax.set_xticks(x)
ax.set_xticklabels([str(cs) for cs in crop_sizes], fontsize=9)
half_span = bar_width * n_methods / 2
ax.set_xlim(x[0] - half_span - 0.02, x[-1] + half_span + 0.02)
ax.set_ylim(y_lo, y_hi)
ax.set_xlabel("Crop Size (pixels)", fontsize=10)
ax.set_ylabel("MAE (days)", fontsize=10)

# Fine-grained y grid: major (labelled) every 1.0, minor every 0.5
ax.yaxis.set_major_locator(plt.MultipleLocator(1.0))
ax.yaxis.set_minor_locator(plt.MultipleLocator(0.5))
ax.grid(axis="y", which="major", linestyle="-", linewidth=0.6, alpha=0.5, color="#888888")
ax.grid(axis="y", which="minor", linestyle=":", linewidth=0.4, alpha=0.4, color="#aaaaaa")
ax.tick_params(axis="y", which="major", labelsize=9)
ax.set_axisbelow(True)

# Legend below the plot, single row of 5
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), fontsize=8, ncol=5, frameon=True,
          handlelength=1.0, handletextpad=0.4, columnspacing=0.9,
          borderpad=0.3)

plt.tight_layout(pad=0.3)
fig.savefig("paper_latex/Images/crop_size_effect.pdf", bbox_inches="tight", dpi=300)
print("Saved: paper_latex/Images/crop_size_effect.pdf")
plt.show()


## J. Figure 8 — MAE vs. number of missing timesteps (`fig:missing_timesteps`)

`n_missing_ts` counts, per pixel, the input months whose six bands are all zero. Within each
seed, pixels are grouped by tile-year and missing count, averaged within each group
(groups with < 25 pixels are dropped), then averaged equally over tile-years; lines show
the mean across seeds and error bars the std. Only missing counts present in at least five
tile-years are shown (up to 1, 2, 3 missing for T = 4, 8, 12). Gray bars: number of
contributing tile-years.


In [ ]:
methods_missing = {
    "Temporal Transformer": "transformer_1d_paper_nl3_1.0_test",
    "Presto":               "presto_1.0_test",
    "Prithvi":              "prithvi_final_100m_crop32_1.0_test",
}

n_months_list = sorted(MONTH_SUBSETS.keys())
subset_labels = {4: "4 Months", 8: "8 Months", 12: "12 Months"}
MAX_MISSING_BY_MONTHS = {4: 1, 8: 2, 12: 3}

MIN_PIXELS_PER_TILE_BIN = 25
MIN_TILES = 5

# Build missing-timestep data for all subsets
all_missing_dfs = {}
for n_months in n_months_list:
    missing_rows = []
    for method_label, method_key in methods_missing.items():
        seed_dfs = all_results_per_seed[n_months].get(method_key, [])
        if not seed_dfs:
            continue

        # First average pixels within each tile-year and missingness bin.
        seed_tile_rows = []
        for seed_idx, seed_df in enumerate(seed_dfs):
            df = seed_df.copy()
            for d_short in DATE_SHORT:
                valid = df[f"{d_short}_truth_DOY"] >= 0
                df[f"{d_short}_AE"] = np.where(
                    valid,
                    np.abs(df[f"{d_short}_pred_DOY"] - df[f"{d_short}_truth_DOY"]),
                    np.nan,
                )

            tile_cols = ["HLStile", "SiteID", "years", "n_missing_ts"]
            for tile_key, grp in df.groupby(tile_cols):
                tile, siteid, year, n_miss = tile_key
                date_maes = {d_name: grp[f"{d_short}_AE"].mean()
                             for d_name, d_short in zip(DATE_NAMES, DATE_SHORT)}
                seed_tile_rows.append({
                    "seed": seed_idx, "HLStile": tile, "SiteID": siteid,
                    "years": year, "n_missing_ts": int(n_miss),
                    "n_pixels": len(grp), **date_maes,
                    "Mean": np.nanmean(list(date_maes.values())),
                })

        seed_tile_df = pd.DataFrame(seed_tile_rows)
        seed_tile_df = seed_tile_df[
            seed_tile_df["n_pixels"] >= MIN_PIXELS_PER_TILE_BIN
        ].copy()
        # Then average tile-years within each seed, followed by seeds.
        for n_miss, miss_df in seed_tile_df.groupby("n_missing_ts"):
            n_tiles = miss_df[["HLStile", "SiteID", "years"]].drop_duplicates().shape[0]
            n_pixels = int(miss_df[miss_df["seed"] == 0]["n_pixels"].sum())
            for date_name in DATE_NAMES + ["Mean"]:
                seed_maes = miss_df.groupby("seed")[date_name].mean()
                missing_rows.append({
                    "Method": method_label, "Date": date_name,
                    "n_missing_ts": int(n_miss),
                    "MAE": seed_maes.mean(),
                    "STD": seed_maes.std(ddof=1) if len(seed_maes) > 1 else 0.0,
                    "n_tiles": n_tiles, "n_pixels": n_pixels,
                    "n_seeds": len(seed_maes),
                })
    all_missing_dfs[n_months] = pd.DataFrame(missing_rows)


In [ ]:
# --- Plot (match main-figure style) ---
sns.set_theme(style="whitegrid")

# Same colors as the main figure: Set2 palette indexed by model role
set2 = sns.color_palette("Set2", 4)
method_colors = {
    "Temporal Transformer": set2[0],  # green
    "Presto":               set2[1],  # orange
    "Prithvi":              set2[2],  # blue
}

fig, axes = plt.subplots(1, len(n_months_list),
                         figsize=(6 * len(n_months_list), 5), sharey=True)

for col_idx, n_months in enumerate(n_months_list):
    ax = axes[col_idx]
    ax2 = ax.twinx()
    ax2.grid(False)
    mdf_miss = all_missing_dfs[n_months]

    first_method = list(methods_missing.keys())[0]
    count_sub = mdf_miss[(mdf_miss["Method"] == first_method) & (mdf_miss["Date"] == "Mean")].sort_values("n_missing_ts")
    count_sub = count_sub[count_sub["n_tiles"] >= MIN_TILES]
    count_sub = count_sub[
        count_sub["n_missing_ts"] <= MAX_MISSING_BY_MONTHS[n_months]
    ]
    valid_ts = sorted(count_sub["n_missing_ts"].values)
    max_miss = max(valid_ts) if valid_ts else 0

    if len(count_sub) > 0:
        ax2.bar(count_sub["n_missing_ts"], count_sub["n_tiles"],
                alpha=0.15, color="gray", width=0.8, zorder=1)
        if col_idx == len(n_months_list) - 1:
            ax2.set_ylabel("# Tile-years", fontsize=14, color="gray")
        ax2.tick_params(axis="y", labelsize=12, labelcolor="gray")
        if col_idx < len(n_months_list) - 1:
            ax2.set_yticklabels([])

    for method_label in methods_missing.keys():
        m_sub = mdf_miss[(mdf_miss["Method"] == method_label) & (mdf_miss["Date"] == "Mean")].sort_values("n_missing_ts")
        m_sub = m_sub[m_sub["n_missing_ts"].isin(valid_ts)]
        if len(m_sub) == 0:
            continue
        ax.errorbar(m_sub["n_missing_ts"], m_sub["MAE"], yerr=m_sub["STD"], marker="o",
                label=method_label if col_idx == 0 else None,
                color=method_colors[method_label], linewidth=2, markersize=8,
                capsize=3, elinewidth=1, zorder=3)

    ax.set_title(f"{subset_labels[n_months]}", fontsize=16, fontweight="bold")
    ax.set_xlabel("# Missing Timesteps", fontsize=14)
    ax.set_xticks(valid_ts)
    ax.tick_params(axis="x", labelsize=12)
    ax.tick_params(axis="y", labelsize=12)
    ax.set_xlim(-0.5, max_miss + 0.5)

    # Fine-grained y grid: major every 1, minor every 0.5
    ax.yaxis.set_major_locator(plt.MultipleLocator(1))
    ax.yaxis.set_minor_locator(plt.MultipleLocator(0.5))
    ax.grid(axis="y", which="major", linestyle="-", linewidth=0.8, alpha=0.5, color="#888888")
    ax.grid(axis="y", which="minor", linestyle=":", linewidth=0.5, alpha=0.4, color="#aaaaaa")
    ax.set_axisbelow(True)

    if col_idx == 0:
        ax.set_ylabel("Mean MAE (days)", fontsize=14)

fig.legend(*axes[0].get_legend_handles_labels(), loc="lower center",
           ncol=len(methods_missing), bbox_to_anchor=(0.5, -0.06), fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
fig.savefig("paper_latex/Images/missing_timesteps.pdf", bbox_inches="tight", dpi=300)
print("Saved: paper_latex/Images/missing_timesteps.pdf")
